In [2]:
import torch

# Autograd graph

Do the Derivatives manually using chain rule and check with pytorch calculation.

In [3]:
x = torch.tensor(2.0, requires_grad=True) 
w = torch.tensor(3.0, requires_grad=True) 
b = torch.tensor(1.0, requires_grad=True) 

y = w*x + b 
z = y**2 

z.backward() 

print('dz/dz',x.grad) 
print("dz/dw",w.grad) 
print("dz/db",b.grad)

dz/dz tensor(42.)
dz/dw tensor(28.)
dz/db tensor(14.)


- Why does backprop starts from `z`?
- Why don't we call `z.backward(1)` explicitly?

In [ ]:
print(z.grad_fn)
print(y.grad_fn)

- What is `grad_fn`?
- Why does `x.grad_fn` not exist?

#### code modification

In [ ]:
# Modified code
x = torch.tensor(2.0, requires_grad=True) 
w = torch.tensor(3.0, requires_grad=True) 
b = torch.tensor(1.0, requires_grad=True) 

y = w*x + b 

y_detached = y.detach()
z = y_detached**2 
print(z)

z.backward() 

print('dz/dz',x.grad) 
print("dz/dw",w.grad) 
print("dz/db",b.grad)

tensor(49.)


RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

# QnA

1) What is `grad_fn`?

Ans:- grad_fn is a reference to the backward function of the operation that created the tensor

2) Why is the above [error](#code-modification)?
  
Ans:- 
  - `y_detached` has `requires_grad=False`

  - `z` also has `requires_grad=False`

  - PyTorch sees nothing to backpropagate
  
For clarification check 3️⃣ `detach()` cell below.

# Interactions

### 1️⃣ Manual Gradient Check



Let’s recompute together:

Given
```python
x = 2
w = 3
b = 1
y = wx + b = 3*2 + 1 = 7
z = y² = 49
```
Derivatives

- $\frac{dz}{dy} = 2y = 14$ <br>

- $\frac{dx}{dy} = w = 3 \implies \frac{dz}{dx} = 3 \times 14 = 42$ <br>

- $\frac{dy}{dw} = x = 3 \implies \frac{dz}{dw} = 14 \times 2 = 28$ <br>

- $\frac{dy}{db} = 1 \implies \frac{dz}{db} = 14 \times 1 = 14$ <br>

### 2️⃣ grad_fn — Clean Mental Model (IMPORTANT)



“grad_fn is the graph”

#### ✅ Correct mental model

grad_fn is a reference to the backward function of the operation that created the tensor

- Example:
```python
z = y ** 2
print(z.grad_fn)
```
Output:
```
<PowBackward0>
```
Meaning:

- z knows how it was created

- It stores how to compute gradients backward

- Each tensor points to its local backward function

- Together, these form the computation graph

***

#### Why x.grad_fn is **None**?

```python
x = torch.tensor(2.0, requires_grad=True)
print(x.grad_fn)  # None
```
Because:

- x is a leaf tensor

- It was not created by an operation

- Gradients are stored here, not computed here

📌 Interview sentence:

- grad_fn defines how a tensor propagates gradients backward through the computation graph.

### 3️⃣ `detach()`

“detach assigns only the value”

✅ Explanation (precise)

`y_detached = y.detach()`

Means:

- Create a new tensor that shares the same value BUT is disconnected from the computation graph

Key properties:
| Property         | y    | y_detached |
| ---------------- | ---- | ---------- |
| Value            | same | same       |
| requires_grad    | True | False      |
| Graph connection | Yes  | ❌ No       |


#### ✅ Real-world reasons detach() is used


1. Stop gradient flow intentionally

Example: GANs

- Generator loss should not update discriminator

- So discriminator output is detached

***

2. Feature extraction
```python
features = backbone(images).detach()
```
Backbone frozen

Classifier trained on fixed features

***

1. Logging / monitoring
   
```python
loss_value = loss.detach().cpu().item()
```
Avoid graph retention

Prevent memory leaks

***

1. Target networks (RL)

Stable targets

No gradient leakage
***

📌 Correct refined answer you should remember:

- `detach()` is used to intentionally stop gradient propagation while preserving tensor values.

# Roughs